# Task 3: Domain Adaptation / Fine-Tuning (LoRA)

Small-scale LoRA fine-tuning experiment on top of Task 1's setup, to establish a baseline domain-adaptation workflow.

- **Dataset:** [RSICD](https://huggingface.co/datasets/arampacha/rsicd) — same as Task 1. A tiny random subset (~200 images) is used for LoRA fine-tuning; the rest form a held-out evaluation sample (with the fine-tuning images excluded).
- **Base model:** generic CLIP `ViT-B-32` (`laion2b_s34b_b79k`) — the *not* domain-adapted checkpoint Task 1 skipped past in favor of RemoteCLIP.
- **Approach:** LoRA fine-tune the base model for **1 epoch** on the tiny subset, with a standard CLIP contrastive loss. Compare text→image retrieval on the held-out sample across three models: base (untuned) CLIP, LoRA fine-tuned CLIP, and RemoteCLIP (fully domain-adapted reference).
- **Flow:** load dataset → pick FT subset + held-out eval sample → load base CLIP → eval baseline → apply LoRA → fine-tune (1 epoch) → eval after fine-tuning → load RemoteCLIP → eval reference → findings write-up (bottom of notebook).

### Step 1: Setup

In [ ]:
%pip install open_clip_torch peft

In [ ]:
import re
import random
from collections import defaultdict

import torch
import torch.nn.functional as F
import open_clip

from datasets import load_dataset
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model

### Step 2: Load RSICD, pick a tiny LoRA fine-tuning subset, and build a held-out evaluation sample

The fine-tuning subset and the evaluation sample must not overlap, otherwise "held-out" comparison would be meaningless.

In [ ]:
ds = load_dataset("arampacha/rsicd")
ds

In [ ]:
random.seed(0)

# Very small LoRA fine-tuning subset, per the "small subset, one epoch" scope for this task.
NUM_FT_SAMPLES = 200

ft_indices = random.sample(range(len(ds["train"])), NUM_FT_SAMPLES)
ft_indices_set = set(ft_indices)

print(f"Fine-tuning subset: {len(ft_indices)} images")

In [ ]:
# Same category-from-filename convention as Task 1: only "category_number.jpg"
# filenames carry a reliable category label; plain numeric filenames are excluded.
CATEGORY_RE = re.compile(r'^([a-zA-Z]+)_\d+\.jpg$')

def category_from_filename(filename):
    name = filename.split('/')[-1]
    match = CATEGORY_RE.match(name)
    return match.group(1) if match else None

by_category = defaultdict(list)
uncategorized = 0
for idx, fname in enumerate(ds['train']['filename']):
    if idx in ft_indices_set:
        continue  # keep the eval sample disjoint from the fine-tuning subset
    cat = category_from_filename(fname)
    if cat is None:
        uncategorized += 1
    else:
        by_category[cat].append(idx)

eval_sample_indices = []
for cat, idxs in by_category.items():
    eval_sample_indices.extend(random.sample(idxs, min(5, len(idxs))))

print(f"Held-out eval sample: {len(eval_sample_indices)} images across {len(by_category)} labeled categories")
print(f"(Excludes the {len(ft_indices)} fine-tuning images and {uncategorized} unlabeled images)")

### Step 3: Load the base (generic, non-domain-adapted) CLIP model

In [ ]:
model_name = "ViT-B-32"

base_model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained="laion2b_s34b_b79k")
tokenizer = open_clip.get_tokenizer(model_name)
base_model = base_model.cuda().eval()

### Step 4: Evaluation harness

Fixed set of novel text queries (same style as Task 1), each with the RSICD category label(s) that would count as a correct top-3 retrieval. Reused as-is across all three models below so the comparison is apples-to-apples.

In [ ]:
queries_with_expected = [
    ("a large bridge crossing a river", ["bridge"]),
    ("an airport with parked airplanes", ["airport"]),
    ("a dense green forest", ["forest"]),
    ("farmland divided into rectangular plots", ["farmland"]),
    ("a stadium with a running track", ["stadium"]),
    ("a residential area with many houses", ["denseresidential", "mediumresidential", "sparseresidential"]),
]
queries = [q for q, _ in queries_with_expected]

def evaluate_retrieval(model, label):
    model.eval()
    sample_images = [ds['train'][i]['image'] for i in eval_sample_indices]
    sample_filenames = [ds['train'][i]['filename'] for i in eval_sample_indices]
    sample_categories = [category_from_filename(f) for f in sample_filenames]

    image_tensors = torch.stack([preprocess(img) for img in sample_images]).to("cuda")

    with torch.no_grad(), torch.autocast("cuda"):
        bank_image_features = model.encode_image(image_tensors)
        bank_image_features /= bank_image_features.norm(dim=-1, keepdim=True)

        query_tokens = tokenizer(queries).to("cuda")
        query_features = model.encode_text(query_tokens)
        query_features /= query_features.norm(dim=-1, keepdim=True)

    sims = query_features @ bank_image_features.T

    print(f"\n{'=' * 20} {label} {'=' * 20}")
    hits = 0
    for qi, (q, expected_cats) in enumerate(queries_with_expected):
        top_idx = sims[qi].topk(3).indices.tolist()
        top_cats = [sample_categories[ii] for ii in top_idx]
        hit = any(c in expected_cats for c in top_cats)
        hits += hit
        mark = "\u2713" if hit else "\u2717"
        print(f"  {mark} {q!r} -> top3 categories: {top_cats}")
    print(f"  Score: {hits}/{len(queries)} queries had an expected category in their top 3")
    return hits

### Step 5: Baseline — evaluate the untuned base CLIP model

In [ ]:
base_score = evaluate_retrieval(base_model, "Base generic CLIP (untuned)")

### Step 6: Apply LoRA to the base model

Target modules match open_clip's `ResidualAttentionBlock` implementation (shared by both the vision and text towers): `out_proj` is the `nn.Linear` output projection inside each `nn.MultiheadAttention`, and `c_fc`/`c_proj` are the two MLP linear layers.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["out_proj", "c_fc", "c_proj"],
    lora_dropout=0.05,
    bias="none",
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

# Unwrap to the underlying CLIP model (now with LoRA layers injected in place) so we can
# keep calling the familiar .encode_image()/.encode_text()/.logit_scale interface directly.
clip = lora_model.base_model.model

### Step 7: Fine-tune with LoRA (1 epoch, tiny subset)

Standard CLIP contrastive loss (symmetric cross-entropy over the in-batch image–text similarity matrix). Only the LoRA adapter weights are trainable — the base CLIP weights stay frozen.

In [ ]:
BATCH_SIZE = 16
LR = 1e-4

optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=LR)

shuffled_ft_indices = ft_indices.copy()
random.shuffle(shuffled_ft_indices)

clip.train()
losses = []
num_batches = (len(shuffled_ft_indices) + BATCH_SIZE - 1) // BATCH_SIZE

for b, start in enumerate(range(0, len(shuffled_ft_indices), BATCH_SIZE), 1):
    batch_idx = shuffled_ft_indices[start:start + BATCH_SIZE]
    batch_images = torch.stack([preprocess(ds['train'][i]['image']) for i in batch_idx]).to("cuda")
    batch_captions = [ds['train'][i]['captions'][0] for i in batch_idx]
    batch_text = tokenizer(batch_captions).to("cuda")

    with torch.autocast("cuda"):
        image_features = clip.encode_image(batch_images)
        text_features = clip.encode_text(batch_text)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        logit_scale = clip.logit_scale.exp()
        logits_per_image = logit_scale * image_features @ text_features.T
        logits_per_text = logits_per_image.T

        labels = torch.arange(len(batch_idx), device="cuda")
        loss = (F.cross_entropy(logits_per_image, labels) + F.cross_entropy(logits_per_text, labels)) / 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    print(f"batch {b}/{num_batches}  loss={loss.item():.4f}")

print(f"\nMean loss over the epoch: {sum(losses) / len(losses):.4f}")

### Step 8: Evaluate after LoRA fine-tuning

In [ ]:
lora_score = evaluate_retrieval(clip, f"LoRA fine-tuned CLIP (1 epoch, {NUM_FT_SAMPLES} samples)")

### Step 9: Load RemoteCLIP as the fully domain-adapted reference point

In [ ]:
checkpoint_path = hf_hub_download("chendelong/RemoteCLIP", f"RemoteCLIP-{model_name}.pt", cache_dir="checkpoints")

remoteclip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=None)
ckpt = torch.load(checkpoint_path, map_location="cpu")
message = remoteclip_model.load_state_dict(ckpt)
print(message)

remoteclip_model = remoteclip_model.cuda().eval()

In [ ]:
remoteclip_score = evaluate_retrieval(remoteclip_model, "RemoteCLIP (fully domain-adapted reference)")

In [ ]:
print("Summary (score = queries with an expected category in top-3, out of", len(queries), ")")
print(f"  Base generic CLIP (untuned):  {base_score}")
print(f"  LoRA fine-tuned CLIP:         {lora_score}")
print(f"  RemoteCLIP (reference):       {remoteclip_score}")

## Step 10: Findings write-up

_TBD — to be filled in after running all cells and reviewing the actual output._